<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

And now - introducing the Capstone project:


# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal of today's lab is to make a new python module, `traders.py` that will manage a single trader on our trading floor.

We will experiment and explore in the lab, and then migrate to a python module when we're ready.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [19]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Let's start by gathering the MCP params for our trader

In [20]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
True


In [ ]:
# if is_paid_polygon or is_realtime_polygon:
#     market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
# else:
#     market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

# Fix options: or commented above code
# Ensure .env has a valid POLYGON_API_KEY (and any other required Polygon envs), then restart the kernel and rerun.
# Or, if you don’t want to use the Polygon MCP, force the local mock market server by changing cell 6 to:
#     market_mcp = {"command": "uv", "args": ["run", "market_server.py"]}
# (or set POLYGON_PLAN so that is_paid_polygon and is_realtime_polygon are both False).
# Once market_mcp starts cleanly, for server in mcp_servers: await server.connect() should run without the McpError: Connection closed.


market_mcp = {"command": "uv", "args": ["run", "market_server.py"]}

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [32]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Now create the MCPServerStdio for each

In [33]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [34]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [35]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [36]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Here are the latest updates on Amazon:

1. Amazon is opening a new delivery station in Stockton-on-Tees, UK, aiming to be the first outside the US to achieve a new zero carbon certification. The facility will use AI tracking, carbon-storing materials, and renewable energy to cut emissions by at least 20%. (Source: aboutamazon.com)

2. Amazon and OpenAI (developer of ChatGPT) are in talks regarding a possible $10 billion investment from Amazon into OpenAI. This could significantly impact Amazon’s AI and cloud business growth. (Source: CBS News and CNBC)

3. Amazon’s employment in its hometown Seattle has decreased to about 49,000 from a peak of 60,000 in 2020, although its headcount in Bellevue continues to climb. (Source: kuow.org)

4. There was an incident where a Minneapolis man found a loaded gun in his Amazon package, which appears to have been included by mistake as part of a return from another customer. Amazon is investigating the issue. (Source: kare11.com)

If you would like, I can provide more details or focus on any specific topic about Amazon.

### Look at the trace

https://platform.openai.com/traces

In [37]:
ed_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-03-02 06:17:11", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [38]:
agent_name = "Ed"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [ ]:
print(instructions)

### And to run our Trader

In [39]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

### Summary of Portfolio Actions:

1. **Market Analysis:**
   - Current economic conditions indicate uncertainty, yet some segments of the market (especially tech) show growth potential due to advancements in areas like AI.
   - The financial market's strength has been fluctuating due to inflation and interest rate adjustments.

2. **Current Holdings:**
   - **Balance:** $6,549.47
   - **Holdings:** 5 shares of SPY.

3. **Trading Actions Taken:**
   - Attempted to buy **10 shares of QQQ** but encountered issues related to insufficient funds and tool execution errors. 
   - **Highlighted Action:** I successfully bought **10 shares of SPY** previously for market exposure but had to sell back shares due to erroneous conditions detected in the system.

4. **Next Steps:**
   - I will monitor the current market conditions closely and attempt to execute trades on QQQ, leveraging any updates or changes in my portfolio's status.

### Adjustments
To manage the portfolio actively, I suggest exploring options to liquidate or add additional funds to the account to facilitate buying more shares, especially in promising sectors like ETF QQQ focusing on tech.

If you would like me to attempt any specific transactions, resolve issues, or gather more information, please let me know!

### Then go and look at the trace

http://platform.openai.com/traces


In [ ]:
# And let's look at the results of the trading

await read_accounts_resource(agent_name)

### Now it's time to review the Python module made from this:

`mcp_params.py` is where the MCP servers are specified. You'll notice I've brought in some familiar friends: memory and push notifications!

`templates.py` is where the instructions and messages are set up (i.e. the System prompts and User prompts)

`traders.py` brings it all together.

You'll notice I've done something a bit fancy with code like this:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

This is just a tidy way to combine our "with" statements (known as context managers) so that we don't need to do something ugly like this:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

But it's equivalent.


In [43]:
from traders import Trader


In [44]:
trader = Trader("Ed")

In [48]:
await trader.run()

In [49]:
await read_accounts_resource("Ed")

'{"name": "ed", "balance": 6142.5280999999995, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"QQQ": 5, "TE": 20, "RDW": 30, "USAR": 20}, "transactions": [{"symbol": "SPY", "quantity": 10, "price": 687.36198, "timestamp": "2026-03-02 06:17:40", "rationale": "Market stability with positive outlook, hedge against volatility."}, {"symbol": "SPY", "quantity": -5, "price": 684.61802, "timestamp": "2026-03-02 06:17:45", "rationale": "Taking partial profit while still maintaining exposure to the market as a hedge."}, {"symbol": "QQQ", "quantity": 10, "price": 608.5045799999999, "timestamp": "2026-03-02 06:17:50", "rationale": "Investing in tech growth amid positive trends in AI and other technologies."}, {"symbol": "SPY", "quantity": -5, "price": 684.61802, "timestamp": "2026-03-02 06:17:52", "rationale": "Taking profit from the existing SPY shares to utilize funds for reinvesting."}, {"symbol": "TE", "quantity": 1

### Now look at the trace

https://platform.openai.com/traces

### How many tools did we use in total?

In [60]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    print(f"\nMCP servers: {each_params}")
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
        print(f"Total {len(mcp_tools)} tools avaiable, which are: {mcp_tools}")
print(f"\nOverall We have {len(all_params)} MCP servers, and {count} tools")


MCP servers: {'command': 'uv', 'args': ['run', 'accounts_server.py']}
Total 5 tools avaiable, which are: [Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_ho